# Tuning con Optuna de los 5 modelos (LogReg, RF, XGBoost, LightGBM, CatBoost)

Re-tunea los tres modelos del pipeline sobre el set de features vigente (71 variables, con las
de campañas) y tunea **por primera vez** LightGBM y CatBoost, todos bajo el mismo protocolo que
`05_modelling/0{2,3,4}_tuning_*_optuna.ipynb`:

- **Nested CV**: Optuna maximiza el AUC GroupKFold(5) por vendedora **dentro del train-pool**
  (= train del split OOT). El bloque OOT (últimos 4 meses, gap 6) no se toca durante el tuning.
- **Evaluación de los mejores hiperparámetros** con los dos protocolos del repo: GroupKFold(5)
  sobre todo el dataset (métrica principal) y OOT (entrena en train, predice test una sola vez).
- También se evalúa la **config previa** de cada modelo (hiperparámetros del repo / defaults de
  `11_ensemble.py`) con el mismo protocolo, para medir cuánto aporta el tuning.
- Desbalance: `class_weight="balanced"` / `scale_pos_weight` / `auto_class_weights="Balanced"`.

**Salidas** en `resultados/` (se guardan al terminar cada modelo):
`resultados.md|csv` (tabla comparativa), `<modelo>_best_params.json` (mismo formato que
`05_modelling/*_best_params.json`), `modelos/<modelo>_tuned.joblib` (fit en el train del OOT, como
`models/*_tuned.joblib`), `trials_<modelo>.csv`, `predicciones.csv` (OOF GroupKFold + OOT por modelo,
para ensembles sin re-entrenar) y `progreso.log`.

**Reanudable**: cada estudio vive en `resultados/optuna_<modelo>.journal`. Si la corrida se corta,
volver a ejecutar el notebook retoma donde quedó (solo corre los trials que faltan).

`SMOKE=1` en el entorno → 3 trials por modelo y salidas en `resultados_smoke/` (prueba de instalación).

In [ ]:
import json
import os
import platform
import time
import warnings
from pathlib import Path

import catboost
import joblib
import lightgbm
import numpy as np
import optuna
import pandas as pd
import sklearn
import xgboost
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=ConvergenceWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- Configuración: lo único que hace falta tocar ------------------------------
MODELOS = ["logreg", "rf", "xgboost", "lightgbm", "catboost"]   # orden de ejecución
N_TRIALS = {"logreg": 100, "rf": 300, "xgboost": 500, "lightgbm": 500, "catboost": 400}
THREADS_POR_TRIAL = 4                                            # hilos de cada modelo
N_PARALELO = max(1, (os.cpu_count() or 1) // THREADS_POR_TRIAL)   # trials simultáneos
RS = 42

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    N_TRIALS = {m: 3 for m in N_TRIALS}
BASE = Path.cwd()
OUT = BASE / ("resultados_smoke" if SMOKE else "resultados")
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
ENTORNO = {"python": platform.python_version(), "cpus": os.cpu_count(), "n_paralelo": N_PARALELO,
           "sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
           "lightgbm": lightgbm.__version__, "catboost": catboost.__version__,
           "optuna": optuna.__version__}
print(ENTORNO, "| trials:", N_TRIALS)

## 1. Datos y split OOT

`churn_dataset_features.csv` es la salida de `04_feature_engineering&selection` (71 features ya preprocesadas y seleccionadas). Split idéntico a `pipeline.oot_split`: test = últimos 4 meses etiquetados, gap = 6.

In [ ]:
ID = ["id_vendedor", "mes_obs", "mes_rank"]
TARGET = "churn"
df = pd.read_csv(BASE / "data" / "churn_dataset_features.csv", parse_dates=["mes_obs"])
FEATS = [c for c in df.columns if c not in ID + [TARGET]]
X, y, groups = df[FEATS], df[TARGET].values, df["id_vendedor"].values
mes = df["mes_obs"].dt.strftime("%Y-%m").values

rank = df["mes_rank"]
test_start = rank.max() - 4 + 1
train_mask = (rank <= test_start - 1 - 6).values
test_mask = (rank >= test_start).values
assert len(FEATS) == 71 and test_mask.sum() == 885, "no es el dataset vigente (71 features, 885 filas OOT)"

# train-pool del nested CV = train del OOT; los folds del tuning viven solo acá
Xp, yp, gp = X[train_mask].reset_index(drop=True), y[train_mask], groups[train_mask]
FOLDS = list(GroupKFold(5).split(Xp, yp, gp))
print(f"{len(df):,} filas | {len(FEATS)} features | train-pool {train_mask.sum():,} "
      f"| OOT {test_mask.sum()} (prevalencia {y[test_mask].mean():.3f})")

## 2. Modelos y espacios de búsqueda

Rangos más amplios que los notebooks originales donde el óptimo previo quedó en un borde (XGBoost salió con `max_depth=3`, el mínimo, y `learning_rate` cerca del piso). `PREVIOS` = config vigente de cada modelo, para medir el aporte del tuning.

In [ ]:
def make_model(name, p, y_tr, threads):
    spw = (y_tr == 0).sum() / (y_tr == 1).sum()
    if name == "logreg":
        return make_pipeline(StandardScaler(), LogisticRegression(
            **p, class_weight="balanced", solver="saga", max_iter=5000, random_state=RS))
    if name == "rf":
        return RandomForestClassifier(**p, class_weight="balanced", n_jobs=threads, random_state=RS)
    if name == "xgboost":
        return XGBClassifier(**p, scale_pos_weight=spw, eval_metric="logloss", tree_method="hist",
                             n_jobs=threads, random_state=RS)
    if name == "lightgbm":
        return LGBMClassifier(**p, subsample_freq=1, scale_pos_weight=spw, n_jobs=threads,
                              random_state=RS, verbose=-1)
    if name == "catboost":
        return CatBoostClassifier(**p, auto_class_weights="Balanced", random_seed=RS,
                                  thread_count=threads, verbose=0, allow_writing_files=False)
    raise ValueError(name)


def espacio(name, t):
    f, i = t.suggest_float, t.suggest_int
    if name == "logreg":
        return dict(C=f("C", 1e-4, 1e2, log=True), l1_ratio=f("l1_ratio", 0.0, 1.0))
    if name == "rf":
        return dict(n_estimators=i("n_estimators", 200, 1500, step=50),
                    max_depth=i("max_depth", 3, 24),
                    max_features=f("max_features", 0.05, 1.0),
                    min_samples_leaf=i("min_samples_leaf", 1, 200, log=True),
                    min_samples_split=i("min_samples_split", 2, 40),
                    max_samples=f("max_samples", 0.3, 1.0))
    if name == "xgboost":
        return dict(n_estimators=i("n_estimators", 200, 3000, step=50),
                    max_depth=i("max_depth", 2, 8),
                    learning_rate=f("learning_rate", 2e-3, 0.2, log=True),
                    subsample=f("subsample", 0.4, 1.0),
                    colsample_bytree=f("colsample_bytree", 0.3, 1.0),
                    min_child_weight=i("min_child_weight", 1, 100, log=True),
                    gamma=f("gamma", 0.0, 10.0),
                    reg_alpha=f("reg_alpha", 1e-3, 10.0, log=True),
                    reg_lambda=f("reg_lambda", 1e-3, 30.0, log=True))
    if name == "lightgbm":
        return dict(n_estimators=i("n_estimators", 200, 3000, step=50),
                    learning_rate=f("learning_rate", 2e-3, 0.2, log=True),
                    num_leaves=i("num_leaves", 4, 64, log=True),
                    max_depth=i("max_depth", 2, 10),
                    min_child_samples=i("min_child_samples", 5, 300, log=True),
                    subsample=f("subsample", 0.4, 1.0),
                    colsample_bytree=f("colsample_bytree", 0.3, 1.0),
                    min_split_gain=f("min_split_gain", 0.0, 5.0),
                    reg_alpha=f("reg_alpha", 1e-3, 10.0, log=True),
                    reg_lambda=f("reg_lambda", 1e-3, 30.0, log=True))
    if name == "catboost":
        return dict(iterations=i("iterations", 200, 3000, step=50),
                    depth=i("depth", 2, 8),
                    learning_rate=f("learning_rate", 2e-3, 0.2, log=True),
                    l2_leaf_reg=f("l2_leaf_reg", 0.5, 50.0, log=True),
                    random_strength=f("random_strength", 0.0, 10.0),
                    bagging_temperature=f("bagging_temperature", 0.0, 2.0),
                    rsm=f("rsm", 0.3, 1.0),
                    border_count=i("border_count", 32, 254))
    raise ValueError(name)


PREVIOS = {
    'logreg': {'C': 0.03277622927423502, 'l1_ratio': 0.43968429924639924},
    'rf': {'n_estimators': 350, 'max_depth': 10, 'max_features': 0.1638581982754919, 'min_samples_leaf': 28, 'min_samples_split': 6, 'max_samples': 0.590664012982737},
    'xgboost': {'n_estimators': 950, 'max_depth': 3, 'learning_rate': 0.011011952680452656, 'subsample': 0.821741761207133, 'colsample_bytree': 0.8024943946508585, 'min_child_weight': 15, 'gamma': 2.200893624894427, 'reg_alpha': 0.004938288434068096, 'reg_lambda': 0.25855890817792654},
    'lightgbm': {'n_estimators': 800, 'learning_rate': 0.02, 'num_leaves': 15, 'min_child_samples': 30, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0},
    'catboost': {'iterations': 1000, 'depth': 4, 'learning_rate': 0.03, 'l2_leaf_reg': 3},
}

## 3. Tuning (nested CV sobre el train-pool)

Objetivo = AUC OOF GroupKFold(5) dentro del train-pool. Poda con `MedianPruner` sobre el AUC acumulado por fold. Trials en paralelo (`N_PARALELO` hilos × `THREADS_POR_TRIAL`), TPE multivariado con `constant_liar` para no repetir puntos entre trials simultáneos. Con trials en paralelo la corrida no es bit-reproducible.

In [ ]:
def log(msg):
    line = f"{time.strftime('%Y-%m-%d %H:%M:%S')} {msg}"
    print(line, flush=True)
    with open(OUT / "progreso.log", "a") as fh:
        fh.write(line + "\n")


def objective(trial, name):
    p = espacio(name, trial)
    oof = np.zeros(len(yp))
    for k, (tr, va) in enumerate(FOLDS):
        m = make_model(name, p, yp[tr], THREADS_POR_TRIAL).fit(Xp.iloc[tr], yp[tr])
        oof[va] = m.predict_proba(Xp.iloc[va])[:, 1]
        seen = np.concatenate([v for _, v in FOLDS[:k + 1]])
        trial.report(roc_auc_score(yp[seen], oof[seen]), k)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return roc_auc_score(yp, oof)


def tunear(name):
    storage = JournalStorage(JournalFileBackend(str(OUT / f"optuna_{name}.journal")))
    study = optuna.create_study(
        study_name=name, storage=storage, load_if_exists=True, direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RS, multivariate=True, constant_liar=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1))
    for t in study.trials:  # trials que quedaron colgados si la corrida anterior se cortó
        if t.state == TrialState.RUNNING:
            study.tell(t.number, state=TrialState.FAIL)
    faltan = N_TRIALS[name] - sum(t.state.is_finished() for t in study.trials)
    log(f"{name}: {len(study.trials)} trials previos, faltan {max(faltan, 0)}")

    def cb(st, tr):
        done = [t.value for t in st.trials if t.state == TrialState.COMPLETE]
        log(f"{name} #{tr.number} {tr.state.name} "
            f"{'' if tr.value is None else f'{tr.value:.4f}'} | mejor {max(done, default=float('nan')):.4f}")

    if faltan > 0:
        study.optimize(lambda t: objective(t, name), n_trials=faltan, n_jobs=N_PARALELO,
                       callbacks=[cb], gc_after_trial=True)
    return study

## 4. Evaluación con los dos protocolos del repo

- `cv_AUC_trainpool`: mejor valor de Optuna (bucle interno del nested CV; es la métrica de selección, levemente optimista por haber elegido el máximo).
- `gkf_*`: GroupKFold(5) por vendedora sobre **todo** el dataset (protocolo principal, comparable con `07_results/cambios_tras_features_campanas.md` §3).
- `oot_*`: fit en train OOT, predicción del bloque test (escenario de producción). `oot_AUCstd` = std del AUC por mes del bloque test; `*_lift10` = lift del decil top; recall/precision a t=0.5; Brier (calibración).

In [ ]:
def lift10(yt, p):
    n = max(len(yt) // 10, 1)
    return yt[np.argsort(-p)[:n]].mean() / yt.mean()


def evaluar(name, p):
    oof = np.zeros(len(y))
    for tr, va in GroupKFold(5).split(X, y, groups):
        oof[va] = make_model(name, p, y[tr], -1).fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[va])[:, 1]
    model = make_model(name, p, y[train_mask], -1).fit(X[train_mask], y[train_mask])
    pt = model.predict_proba(X[test_mask])[:, 1]
    yt, mt = y[test_mask], mes[test_mask]
    aucs = [roc_auc_score(yt[mt == m], pt[mt == m]) for m in np.unique(mt) if 0 < yt[mt == m].mean() < 1]
    pred = (pt >= 0.5).astype(int)
    met = {"gkf_AUC": roc_auc_score(y, oof), "gkf_PRAUC": average_precision_score(y, oof),
           "gkf_lift10": lift10(y, oof),
           "oot_AUC": roc_auc_score(yt, pt), "oot_AUCstd": float(np.std(aucs)),
           "oot_PRAUC": average_precision_score(yt, pt), "oot_lift10": lift10(yt, pt),
           "oot_rec@0.5": recall_score(yt, pred), "oot_prec@0.5": precision_score(yt, pred, zero_division=0),
           "oot_brier": brier_score_loss(yt, pt)}
    return met, model, oof, pt

## 5. Corrida completa

Por cada modelo: tuning → evaluación del mejor y de la config previa → guarda params, modelo, trials, predicciones y la tabla acumulada. Si algo se corta, lo ya terminado queda guardado en `resultados/`.

In [ ]:
filas = []
preds = df[["id_vendedor", "mes_rank", TARGET]].assign(en_test=test_mask)

for name in MODELOS:
    t0 = time.time()
    study = tunear(name)
    n_ok = sum(t.state == TrialState.COMPLETE for t in study.trials)
    met, model, oof, pt = evaluar(name, study.best_params)
    met_prev, *_ = evaluar(name, PREVIOS[name])

    joblib.dump(model, OUT / "modelos" / f"{name}_tuned.joblib")
    (OUT / f"{name}_best_params.json").write_text(json.dumps({
        "best_params": study.best_params, "best_value_cv_auc_trainpool": study.best_value,
        "oot_auc": met["oot_AUC"], "n_trials": len(study.trials), "n_trials_completos": n_ok,
        "random_state": RS, "metricas": met, "entorno": ENTORNO}, indent=2))
    study.trials_dataframe().to_csv(OUT / f"trials_{name}.csv", index=False)
    preds[f"oof_{name}"] = oof
    preds.loc[test_mask, f"oot_{name}"] = pt
    preds.to_csv(OUT / "predicciones.csv", index=False)

    filas += [{"modelo": name, "config": "tuneado", "trials": len(study.trials),
               "cv_AUC_trainpool": study.best_value, **met, "minutos": (time.time() - t0) / 60},
              {"modelo": name, "config": "previo", **met_prev}]
    res = pd.DataFrame(filas).set_index(["modelo", "config"])
    res.round(4).to_csv(OUT / "resultados.csv")
    log(f"{name} listo en {(time.time() - t0) / 60:.1f} min | cv {study.best_value:.4f} | "
        f"gkf {met['gkf_AUC']:.4f} (previo {met_prev['gkf_AUC']:.4f}) | "
        f"oot {met['oot_AUC']:.4f} (previo {met_prev['oot_AUC']:.4f})")

res.round(4)

## 6. Resumen

Ranking de los modelos tuneados por `gkf_AUC` (protocolo principal). Una diferencia < 0.005 en GroupKFold o menor que `oot_AUCstd` en OOT es ruido.

In [ ]:
tun = res.xs("tuneado", level="config").sort_values("gkf_AUC", ascending=False)
mejor = tun.index[0]
delta = (res.xs("tuneado", level="config") - res.xs("previo", level="config"))[["gkf_AUC", "oot_AUC"]]
lineas = [
    "# Tuning con Optuna — resultados",
    f"\n> {time.strftime('%Y-%m-%d %H:%M')} | entorno: `{ENTORNO}` | trials: `{N_TRIALS}`",
    "\n## Tabla completa (tuneado vs config previa, mismo protocolo)\n", res.round(4).to_markdown(),
    "\n## Ranking de tuneados por gkf_AUC\n",
    tun[["cv_AUC_trainpool", "gkf_AUC", "oot_AUC", "oot_AUCstd", "oot_lift10", "oot_brier"]].round(4).to_markdown(),
    "\n## Aporte del tuning (tuneado − previo)\n", delta.round(4).to_markdown(),
    f"\nMejor por gkf_AUC: **{mejor}** ({tun.loc[mejor, 'gkf_AUC']:.4f} GKF / {tun.loc[mejor, 'oot_AUC']:.4f} OOT).",
]
if len(tun) > 1:
    lineas.append(f"Margen sobre el 2.º ({tun.index[1]}): {tun['gkf_AUC'].iloc[0] - tun['gkf_AUC'].iloc[1]:+.4f} GKF, "
                  f"{tun.loc[mejor, 'oot_AUC'] - tun['oot_AUC'].iloc[1]:+.4f} OOT.")
(OUT / "resultados.md").write_text("\n".join(lineas) + "\n")
print("\n".join(lineas))